# Task 2, Part C5: Information Retention and Systems Cost of the Compressed Caches

* One matched-INT4-budget table (accuracy, reference-answer NLL, post-prefill / teacher-forced KL from FP16, retained
  evidence, packed bytes) and breakdowns by context length, evidence position and distractor count.
* Paired bootstrap over prompts (2,000 resamples) for the principal accuracy differences; accuracy conditional on
  FP16 correctness. The 10 prompts per grid cell are descriptive only.
* Systems: prefill / scoring / compression / decode times and peak allocated memory per policy; independent
  packed-cache encode / decode benchmark at batch 1 and 8 and both lengths (10 warmed-up synchronised repetitions);
  decode-step latency with unpacking vs an FP16 cache at matched shapes. Packed bytes are distinguished from GPU
  allocation: the experiment dequantizes to FP16 before ordinary SDPA.
* A fixed set of aligned generations with evidence spans and FP16 token-selection masks.

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task2_attention_kv")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task2", need_public=False, need_prefix_banks=False)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from shared.src.io import read_jsonl, write_jsonl, save_json
from shared.src.plotting import save_figure
from shared.src.stats import paired_bootstrap_diff, bootstrap_mean_ci
from task2_attention_kv.src import kv_cache as KV
from task2_attention_kv.src import kv_pack as KP

OUT = ctx.results / "kv"
df = pd.DataFrame(read_jsonl(OUT / "per_prompt.jsonl"))
ORDER = ["fp16", "evict_sink_recent", "evict_question", "quant_token_int4", "quant_kivi_int4", "hybrid_fp16_int2"]

## Matched-INT4-budget table (all 180 prompts)

In [ ]:
def agg(g):
    m, lo, hi = bootstrap_mean_ci(g.correct.values)
    return pd.Series(dict(exact_match=m, em_ci=f"[{lo:.3f}, {hi:.3f}]", answer_nll=g.answer_nll.mean(), kl_post_prefill=g.kl_first.mean(),
                          kl_teacher_forced=g.kl_tf_mean.mean(),
                          evidence_retained=g.evidence_retained.mean() if g.name in ("evict_sink_recent", "evict_question", "hybrid_fp16_int2") else np.nan,
                          packed_MiB=g["bytes"].mean() / 2**20, frac_fp16_bytes=g.frac_fp16_bytes.mean(), max_frac_of_budget=(g["bytes"] / g.budget_bytes).max(),
                          k_rel_err=g.k_rel_err.mean(), v_rel_err=g.v_rel_err.mean(),
                          acc_given_ref_correct=g[g.ref_correct == 1].correct.mean()))
main = df.groupby("policy").apply(agg).loc[ORDER]
main.to_csv(OUT / "C5_main_table.csv")
main

## Paired bootstrap of principal accuracy differences (prompts resampled, 2,000 times)

In [ ]:
P = df.pivot(index="id", columns="policy", values="correct")
pairs = [("evict_question", "evict_sink_recent"), ("quant_kivi_int4", "quant_token_int4"), ("hybrid_fp16_int2", "evict_question"),
         ("hybrid_fp16_int2", "quant_kivi_int4"), ("quant_kivi_int4", "evict_question")] + [(p, "fp16") for p in ORDER[1:]]
rows = []
for a, b in pairs:
    m, lo, hi = paired_bootstrap_diff(P[a].values, P[b].values, n_boot=2000)
    rows.append(dict(A=a, B=b, diff_pts=100 * m, ci_lo=100 * lo, ci_hi=100 * hi, n=len(P)))
boot = pd.DataFrame(rows)
boot.to_csv(OUT / "C5_paired_bootstrap.csv", index=False)
boot.round(2)

## Breakdowns by context length, evidence position and distractor count

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), sharey=True)
for ax, fac in zip(axes, ("nominal_len", "position", "distractors")):
    t = df.groupby(["policy", fac]).correct.mean().unstack(fac).loc[ORDER]
    t.T.plot(kind="bar", ax=ax, legend=(fac == "distractors"))
    ax.set_title(f"exact match by {fac}"); ax.set_ylim(0, 1.05)
axes[-1].legend(fontsize=7, loc="lower left")
save_figure(ctx.figures, "kv_accuracy_breakdowns")
for fac in ("nominal_len", "position", "distractors"):
    display(df.pivot_table(index="policy", columns=fac, values=["correct", "answer_nll", "evidence_retained"], aggfunc="mean").loc[ORDER].round(3))

In [ ]:
cell = df.pivot_table(index=["nominal_len", "position", "distractors"], columns="policy", values="correct", aggfunc="mean")[ORDER]
cell.to_csv(OUT / "C5_cells_descriptive.csv")
plt.figure(figsize=(9, 8)); sns.heatmap(cell, annot=True, fmt=".1f", cmap="viridis", vmin=0, vmax=1)
plt.title("Exact match per grid cell (10 prompts each; descriptive)")
save_figure(ctx.figures, "kv_cell_heatmap")

## Systems cost per policy (prefill, scoring, compression, suffix + decode, peak allocated)

In [ ]:
sys_cols = ["prefill_ms", "scoring_ms", "compress_ms", "suffix_ms", "decode_ms", "peak_allocated_mb"]
st = df.groupby(["policy", "nominal_len"])[sys_cols].mean().unstack("nominal_len").loc[ORDER].round(1)
st.to_csv(OUT / "C5_systems_per_policy.csv")
st

In [ ]:
by = df.groupby(["policy", "nominal_len"]).agg(packed_MiB=("bytes", lambda x: x.mean() / 2**20),
                                               fp16_MiB=("fp16_bytes", lambda x: x.mean() / 2**20),
                                               shared_decode_MiB=("shared_decode_bytes", lambda x: x.mean() / 2**20),
                                               peak_alloc_MiB=("peak_allocated_mb", "mean")).loc[ORDER]
by["total_live_cache_MiB"] = by.packed_MiB + by.shared_decode_MiB
by.round(2)

## Packed-cache encode / decode benchmark (real bit packing, 28 layers)

In [ ]:
bench_path = OUT / "C5_pack_benchmark.jsonl"
if not bench_path.exists():
    model, tok = KV.load_kv_model()
    rows = []
    for layout, bits in (("kivi", 4), ("token", 4), ("kivi", 2)):
        for B in (1, 8):
            for L in (2048, 4096):
                r = KP.benchmark_pack(model, B, L, layout, bits, reps=10, warmup=3)
                rows.append(r)
                print(r, flush=True)
                torch.cuda.empty_cache()
    write_jsonl(bench_path, rows)
pb = pd.DataFrame(read_jsonl(bench_path))
pb["formula_bytes"] = [B * KV.bytes_quant(28 * 2 * 128, L, b) for B, L, b in zip(pb.batch, pb.L, pb.bits)]
pb["unpack_overhead_ms"] = pb.decode_step_unpack_ms - pb.decode_step_fp16_ms
pb[["layout", "bits", "batch", "L", "serialized_bytes", "formula_bytes", "ratio", "pack_ms", "unpack_ms", "decode_step_fp16_ms",
    "decode_step_unpack_ms", "unpack_overhead_ms", "peak_alloc_fp16_mb", "peak_alloc_unpack_mb"]].round(3)

## Aligned generations with evidence spans and FP16 selection masks
Fixed rule: the first case (index 00) of each evidence position at 4,096 tokens with 8 distractors.

In [ ]:
sel = pd.DataFrame(read_jsonl(OUT / "selections.jsonl"))
ids = [f"KV-4096-{p}-8-00" for p in ("start", "middle", "end")]
show = df[df.id.isin(ids)][["id", "policy", "code", "pred", "correct", "text", "evidence_retained", "answer_nll"]]
display(show.sort_values(["id", "policy"]))
fig, axes = plt.subplots(len(ids), 1, figsize=(14, 2.2 * len(ids)))
for ax, i in zip(np.atleast_1d(axes), ids):
    s = sel[sel.id == i]
    if s.empty:
        continue
    L = s.iloc[0].ctx_len
    ev = s.iloc[0].evidence
    for y, (_, r) in enumerate(s.iterrows()):
        m = np.zeros(L); m[r.kept_fp16] = 1
        ax.scatter(np.nonzero(m)[0], np.full(int(m.sum()), y), s=0.5, marker="|")
    ax.axvspan(min(ev), max(ev), color="red", alpha=0.3, label="evidence")
    ax.set_yticks(range(len(s))); ax.set_yticklabels(s.policy, fontsize=8); ax.set_title(i, fontsize=9)
ax.set_xlabel("context token position (FP16-retained positions marked)")
save_figure(ctx.figures, "kv_selection_masks")